Loading schema definitions and config

In [0]:
%run ../src/schema_definitions

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
source_file=f"{landing_zone_path}/{batch_id}/Stations.csv"
table_name=f"{catalog}.{bronze_schema}.stations"

Ingest raw dataset into a Spark DataFrame

In [0]:
stations_df=(
    spark.read
    .format("csv")
    .option("header", True)
    .schema(stations_schema)
    .load(source_file) 
)

Add ingestion timestamp, source file name and batch id columns

In [0]:
from pyspark.sql import functions as F

stations_final_df=(
    stations_df.withColumns({
        "ingestion_timestamp" : F.current_timestamp(),
        "source_file" : F.col("_metadata.file_path"),
        "batch_id" : F.lit(batch_id)
    })
)

Write DataFrame to bronze Delta table

In [0]:
stations_write=(
    stations_final_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("batch_id")
    .option("replaceWhere", f"batch_id='{batch_id}'")
    .saveAsTable(table_name)
)